# Day 3 — PySpark Practice Questions: Date & Time Functions

| # | Difficulty | Topic |
|---|-----------|-------|
| 1 | 🟢 Easy    | Find slow deliveries (datediff) |
| 2 | 🟢 Easy    | Monthly revenue totals (date_trunc + groupBy) |
| 3 | 🟡 Medium  | New customers — first order in last 90 days (spark_min + filter) |

In [2]:
import os
import sys
import shutil
from pathlib import Path

# Portable PySpark environment setup: detect current Python and Java automatically.
# Avoid hardcoded instructor or machine-specific paths.
python_exe = sys.executable
os.environ['PYSPARK_PYTHON'] = python_exe
os.environ['PYSPARK_DRIVER_PYTHON'] = python_exe

java_exe = shutil.which('java')
if java_exe:
    java_home = Path(java_exe).resolve().parent.parent
    os.environ['JAVA_HOME'] = str(java_home)
else:
    raise RuntimeError('Java not found on PATH. Install a JDK and add java.exe to PATH, or set JAVA_HOME manually.')

print('Environment set.')
print('JAVA_HOME =', os.environ.get('JAVA_HOME'))
print('PYSPARK_PYTHON =', os.environ.get('PYSPARK_PYTHON'))
print('PYSPARK_DRIVER_PYTHON =', os.environ.get('PYSPARK_DRIVER_PYTHON'))

Environment set.
JAVA_HOME = C:\Users\Friend\AppData\Local\Programs\Eclipse Adoptium\jdk-25.0.3.9-hotspot
PYSPARK_PYTHON = c:\Users\Friend\Downloads\python-pyspark-sql-sessions\myenv\Scripts\python.exe
PYSPARK_DRIVER_PYTHON = c:\Users\Friend\Downloads\python-pyspark-sql-sessions\myenv\Scripts\python.exe


In [3]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit,
    to_date, datediff, date_trunc, date_format,
    year, month, dayofmonth,
    date_add, date_sub,
    months_between, add_months,
    current_date,
    min as spark_min, max as spark_max,
    sum as spark_sum, count, avg,
    round as spark_round,
)
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType,
)

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Day3_PySpark_PQ') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

Spark version: 4.1.2


In [4]:
# Shared dataset — same orders as SQL
data = [
    (1, 1001, '2026-05-01', '2026-05-05', 250.00),
    (2, 1002, '2026-05-03', '2026-05-14', 189.50),
    (3, 1001, '2026-05-10', '2026-05-17', 310.00),
    (4, 1003, '2026-05-15', '2026-05-23', 95.00),
    (5, 1004, '2026-05-20', '2026-05-21', 540.00),
    (6, 1002, '2026-06-01', '2026-06-03', 220.00),
    (7, 1005, '2026-06-05', '2026-06-19', 75.00),
    (8, 1003, '2026-06-10', '2026-06-12', 430.00),
    (9, 1006, '2026-06-12', '2026-06-14', 180.00),
    (10,1007, '2026-06-15', '2026-06-16', 99.00),
    (11,1001, '2025-01-05', '2025-01-07', 500.00),
    (12,1002, '2025-03-12', '2025-03-14', 320.00),
    (13,1003, '2025-06-20', '2025-07-01', 150.00),
    (14,1004, '2025-09-08', '2025-09-10', 780.00),
    (15,1005, '2025-11-25', '2025-11-30', 210.00),
    (16,1001, '2025-12-01', '2025-12-15', 640.00),
    (17,1002, '2025-12-20', '2025-12-22', 110.00),
    (18,1004, '2026-01-10', '2026-01-12', 920.00),
    (19,1001, '2026-02-14', '2026-02-16', 275.00),
    (20,1003, '2026-03-01', '2026-03-04', 360.00),
    (21,1002, '2026-03-18', '2026-03-20', 490.00),
    (22,1005, '2026-04-05', '2026-04-07', 130.00),
]

schema = StructType([
    StructField('order_id',    IntegerType(), False),
    StructField('customer_id', IntegerType(), False),
    StructField('order_date',  StringType(),  False),
    StructField('ship_date',   StringType(),  True),
    StructField('amount',      DoubleType(),  False),
])

df_orders = spark.createDataFrame(data, schema) \
    .withColumn('order_date', to_date(col('order_date'), 'yyyy-MM-dd')) \
    .withColumn('ship_date',  to_date(col('ship_date'),  'yyyy-MM-dd'))

df_orders.printSchema()
df_orders.show(5)

root
 |-- order_id: integer (nullable = false)
 |-- customer_id: integer (nullable = false)
 |-- order_date: date (nullable = false)
 |-- ship_date: date (nullable = true)
 |-- amount: double (nullable = false)

+--------+-----------+----------+----------+------+
|order_id|customer_id|order_date| ship_date|amount|
+--------+-----------+----------+----------+------+
|       1|       1001|2026-05-01|2026-05-05| 250.0|
|       2|       1002|2026-05-03|2026-05-14| 189.5|
|       3|       1001|2026-05-10|2026-05-17| 310.0|
|       4|       1003|2026-05-15|2026-05-23|  95.0|
|       5|       1004|2026-05-20|2026-05-21| 540.0|
+--------+-----------+----------+----------+------+
only showing top 5 rows


---
# Question 1 🟢 — Slow Deliveries

## Concept Warm-up

In [5]:
# Warm-up 1: datediff(end_col, start_col) — end goes FIRST
from pyspark.sql.functions import datediff, col

df_orders.select(
    'order_id', 'order_date', 'ship_date',
    datediff(col('ship_date'), col('order_date')).alias('delivery_days')
).show(5)

+--------+----------+----------+-------------+
|order_id|order_date| ship_date|delivery_days|
+--------+----------+----------+-------------+
|       1|2026-05-01|2026-05-05|            4|
|       2|2026-05-03|2026-05-14|           11|
|       3|2026-05-10|2026-05-17|            7|
|       4|2026-05-15|2026-05-23|            8|
|       5|2026-05-20|2026-05-21|            1|
+--------+----------+----------+-------------+
only showing top 5 rows


In [6]:
# Warm-up 2: WRONG order — what happens when you put start first
df_orders.select(
    'order_id',
    datediff(col('order_date'), col('ship_date')).alias('WRONG_negative_days')
).show(3)

+--------+-------------------+
|order_id|WRONG_negative_days|
+--------+-------------------+
|       1|                 -4|
|       2|                -11|
|       3|                 -7|
+--------+-------------------+
only showing top 3 rows


In [7]:
# Warm-up 3: filter with withColumn then filter
df_check = df_orders \
    .withColumn('delivery_days', datediff(col('ship_date'), col('order_date'))) \
    .filter(col('delivery_days') > 5)
print('Orders with >5 delivery days:', df_check.count())
df_check.select('order_id', 'order_date', 'ship_date', 'delivery_days').show()

Orders with >5 delivery days: 6
+--------+----------+----------+-------------+
|order_id|order_date| ship_date|delivery_days|
+--------+----------+----------+-------------+
|       2|2026-05-03|2026-05-14|           11|
|       3|2026-05-10|2026-05-17|            7|
|       4|2026-05-15|2026-05-23|            8|
|       7|2026-06-05|2026-06-19|           14|
|      13|2025-06-20|2025-07-01|           11|
|      16|2025-12-01|2025-12-15|           14|
+--------+----------+----------+-------------+



## Problem

The logistics team wants to review orders where delivery took **more than 7 days**.  
Return `order_id`, `customer_id`, `order_date`, `ship_date`, and `delivery_days`, sorted slowest first.

In [8]:
# YOUR CODE HERE
df_slow = None
df_slow=df_orders.select(
    col('order_id').alias('order_id'),
    col('order_date'),
    col('ship_date'),
    datediff(col('ship_date'), col('order_date')).alias('delivery_days')
)
df_slow.orderBy(col('delivery_days').desc()).show(5)

+--------+----------+----------+-------------+
|order_id|order_date| ship_date|delivery_days|
+--------+----------+----------+-------------+
|      16|2025-12-01|2025-12-15|           14|
|       7|2026-06-05|2026-06-19|           14|
|       2|2026-05-03|2026-05-14|           11|
|      13|2025-06-20|2025-07-01|           11|
|       4|2026-05-15|2026-05-23|            8|
+--------+----------+----------+-------------+
only showing top 5 rows


---
# Question 2 🟢 — Monthly Revenue Totals

## Concept Warm-up

In [9]:
# Warm-up 1: date_trunc maps all dates to first of month
from pyspark.sql.functions import date_trunc

df_orders.select(
    'order_date',
    date_trunc('month', col('order_date')).alias('month_start')
).distinct().orderBy('month_start').show()

+----------+-------------------+
|order_date|        month_start|
+----------+-------------------+
|2025-01-05|2025-01-01 00:00:00|
|2025-03-12|2025-03-01 00:00:00|
|2025-06-20|2025-06-01 00:00:00|
|2025-09-08|2025-09-01 00:00:00|
|2025-11-25|2025-11-01 00:00:00|
|2025-12-01|2025-12-01 00:00:00|
|2025-12-20|2025-12-01 00:00:00|
|2026-01-10|2026-01-01 00:00:00|
|2026-02-14|2026-02-01 00:00:00|
|2026-03-01|2026-03-01 00:00:00|
|2026-03-18|2026-03-01 00:00:00|
|2026-04-05|2026-04-01 00:00:00|
|2026-05-01|2026-05-01 00:00:00|
|2026-05-03|2026-05-01 00:00:00|
|2026-05-10|2026-05-01 00:00:00|
|2026-05-15|2026-05-01 00:00:00|
|2026-05-20|2026-05-01 00:00:00|
|2026-06-01|2026-06-01 00:00:00|
|2026-06-05|2026-06-01 00:00:00|
|2026-06-10|2026-06-01 00:00:00|
+----------+-------------------+
only showing top 20 rows


In [11]:
# Warm-up 2: date_format produces a sortable string label
from pyspark.sql.functions import date_format

df_orders.select(
    'order_date',
    date_format(col('order_date'), 'yyyy-MM').alias('ym'),
    date_format(col('order_date'), 'MMM yyyy').alias('label')
).show(5)

+----------+-------+--------+
|order_date|     ym|   label|
+----------+-------+--------+
|2026-05-01|2026-05|May 2026|
|2026-05-03|2026-05|May 2026|
|2026-05-10|2026-05|May 2026|
|2026-05-15|2026-05|May 2026|
|2026-05-20|2026-05|May 2026|
+----------+-------+--------+
only showing top 5 rows


In [14]:
# Warm-up 3: groupBy + agg pattern
from pyspark.sql.functions import sum, count

df_orders \
    .groupBy(date_trunc('month', col('order_date')).alias('month_ts')) \
    .agg(count('*').alias('num_orders')) \
    .orderBy('month_ts') \
    .show()

+-------------------+----------+
|           month_ts|num_orders|
+-------------------+----------+
|2025-01-01 00:00:00|         1|
|2025-03-01 00:00:00|         1|
|2025-06-01 00:00:00|         1|
|2025-09-01 00:00:00|         1|
|2025-11-01 00:00:00|         1|
|2025-12-01 00:00:00|         2|
|2026-01-01 00:00:00|         1|
|2026-02-01 00:00:00|         1|
|2026-03-01 00:00:00|         2|
|2026-04-01 00:00:00|         1|
|2026-05-01 00:00:00|         5|
|2026-06-01 00:00:00|         5|
+-------------------+----------+



## Problem

Build a **monthly revenue report** with: `month` (as `'yyyy-MM'`), `total_revenue`, `num_orders`, `avg_order_value`,  
ordered chronologically from oldest to most recent.

In [30]:
from pyspark.sql.functions import col, date_format, sum, count, avg, round

df_monthly = (
    df_orders
    .groupBy(date_format(col('order_date'), 'yyyy-MM').alias('month'))
    .agg(
        sum('amount').alias('total_revenue'),
        count('*').alias('num_orders'),
        round(avg('amount'), 2).alias('avg_amount')
    )
    .orderBy('month')
)

df_monthly.show()

+-------+-------------+----------+----------+
|  month|total_revenue|num_orders|avg_amount|
+-------+-------------+----------+----------+
|2025-01|        500.0|         1|     500.0|
|2025-03|        320.0|         1|     320.0|
|2025-06|        150.0|         1|     150.0|
|2025-09|        780.0|         1|     780.0|
|2025-11|        210.0|         1|     210.0|
|2025-12|        750.0|         2|     375.0|
|2026-01|        920.0|         1|     920.0|
|2026-02|        275.0|         1|     275.0|
|2026-03|        850.0|         2|     425.0|
|2026-04|        130.0|         1|     130.0|
|2026-05|       1384.5|         5|     276.9|
|2026-06|       1004.0|         5|     200.8|
+-------+-------------+----------+----------+



---
# Question 3 🟡 — New Customers (First Order in Last 90 Days)

## Concept Warm-up

In [32]:
# Warm-up 1: current_date() and date_sub to get 90-day cutoff
from pyspark.sql.functions import current_date, date_sub

spark.range(1).select(
    current_date().alias('today'),
    date_sub(current_date(), 90).alias('ninety_days_ago')
).show()

+----------+---------------+
|     today|ninety_days_ago|
+----------+---------------+
|2026-06-23|     2026-03-25|
+----------+---------------+



In [31]:
spark.range(1).select(
    current_date().alias('today'),
    date_sub(current_date(),90).alias('ninety_days_ago')
).show()

+----------+---------------+
|     today|ninety_days_ago|
+----------+---------------+
|2026-06-23|     2026-03-25|
+----------+---------------+



In [33]:
# Warm-up 2: spark_min to find first order date per customer
from pyspark.sql.functions import min as spark_min

df_first = df_orders.groupBy('customer_id') \
    .agg(spark_min('order_date').alias('first_order_date'))
df_first.orderBy('first_order_date').show()

+-----------+----------------+
|customer_id|first_order_date|
+-----------+----------------+
|       1001|      2025-01-05|
|       1002|      2025-03-12|
|       1003|      2025-06-20|
|       1004|      2025-09-08|
|       1005|      2025-11-25|
|       1006|      2026-06-12|
|       1007|      2026-06-15|
+-----------+----------------+



In [34]:
from pyspark.sql.functions import min as spark_min
df_orders.groupBy('customer_id')\
.agg(spark_min('order_date').alias('first_order_date'))\
.orderBy(col('first_order_date'))\
.show()

+-----------+----------------+
|customer_id|first_order_date|
+-----------+----------------+
|       1001|      2025-01-05|
|       1002|      2025-03-12|
|       1003|      2025-06-20|
|       1004|      2025-09-08|
|       1005|      2025-11-25|
|       1006|      2026-06-12|
|       1007|      2026-06-15|
+-----------+----------------+



In [35]:
# Warm-up 3: filter a grouped result by date comparison
# Orders placed in the last 90 days:
df_recent_rows = df_orders.filter(
    col('order_date') >= date_sub(current_date(), 90)
)
print('Recent order rows:', df_recent_rows.count())
df_recent_rows.select('order_id', 'customer_id', 'order_date').show()

Recent order rows: 11
+--------+-----------+----------+
|order_id|customer_id|order_date|
+--------+-----------+----------+
|       1|       1001|2026-05-01|
|       2|       1002|2026-05-03|
|       3|       1001|2026-05-10|
|       4|       1003|2026-05-15|
|       5|       1004|2026-05-20|
|       6|       1002|2026-06-01|
|       7|       1005|2026-06-05|
|       8|       1003|2026-06-10|
|       9|       1006|2026-06-12|
|      10|       1007|2026-06-15|
|      22|       1005|2026-04-05|
+--------+-----------+----------+



In [37]:
# Warm-up 4: datediff to compute days since first order
df_first_with_age = df_first.withColumn(
    'days_since_first',
    datediff(current_date(), col('first_order_date'))
)
df_first_with_age.orderBy('days_since_first').show()

+-----------+----------------+----------------+
|customer_id|first_order_date|days_since_first|
+-----------+----------------+----------------+
|       1007|      2026-06-15|               8|
|       1006|      2026-06-12|              11|
|       1005|      2025-11-25|             210|
|       1004|      2025-09-08|             288|
|       1003|      2025-06-20|             368|
|       1002|      2025-03-12|             468|
|       1001|      2025-01-05|             534|
+-----------+----------------+----------------+



## Problem

The growth team defines a **new customer** as someone whose **first-ever order** was placed within the last 90 days.  
Return `customer_id`, `first_order_date`, and `days_since_first_order`, sorted most-recent first.

In [39]:
# YOUR CODE HERE
df_new_customers =df_first_with_age.filter(col('days_since_first')<=90).show()

+-----------+----------------+----------------+
|customer_id|first_order_date|days_since_first|
+-----------+----------------+----------------+
|       1006|      2026-06-12|              11|
|       1007|      2026-06-15|               8|
+-----------+----------------+----------------+

